# Deurganckdock Oil Slick Detection — Sentinel-1 SLC Burst

An oil slick formed in the **Deurganckdock** (Port of Antwerp) on the night of **9–10 April 2026**.

We use **Sentinel-1 IW SLC-BURST** data from the Alaska Satellite Facility (ASF) to detect the slick by comparing the closest SAR acquisitions before and after the event.

### Why VV polarization?
Oil slicks dampen short-gravity and capillary ocean surface waves, which reduces **Bragg scattering** — the dominant backscatter mechanism over water at C-band. The **VV co-polarization** channel has a better signal-to-noise ratio than VH for detecting this effect, making it the standard choice for oil-spill monitoring (Brekke & Solberg 2005; Bianchi et al. 2020).

We keep the data at **full SLC burst resolution** (~5 × 20 m) to maximise spatial detail for slick delineation.

## 1. Import Required Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import asf_search
import rasterio

# Core RS-tools modules (reuse existing slider comparison)
from rs_tools.visualization.slider import slider_comparison

# Output directory (follows repository convention)
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(os.path.join(OUT_DIR, "bursts"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "gifs"), exist_ok=True)

print("Libraries loaded")

## 2. Define Area of Interest and Event Parameters

The **Deurganckdock** is a large tidal dock in the Port of Antwerp on the left bank of the Scheldt.  
We define a tight bounding box around the dock and search for Sentinel-1 passes within ±2 weeks of the event to guarantee at least one acquisition on each side (the S1 repeat cycle is 6–12 days).

In [ ]:
# Deurganckdock, Antwerp port
BBOX = [
    4.251283836960951,   # west
    51.28632393164611,   # south
    4.271149638243543,   # east
    51.298511604339325,  # north
]

EVENT_DATE   = "2026-04-10"   # Oil slick first observed
SEARCH_START = "2026-03-25"   # ~2 weeks before
SEARCH_END   = "2026-04-25"   # ~2 weeks after

# WKT polygon for ASF search
WKT = (
    f"POLYGON(({BBOX[0]} {BBOX[1]},{BBOX[2]} {BBOX[1]},"
    f"{BBOX[2]} {BBOX[3]},{BBOX[0]} {BBOX[3]},{BBOX[0]} {BBOX[1]}))"
)

print(f"AOI:          {BBOX}")
print(f"Event date:   {EVENT_DATE}")
print(f"Search range: {SEARCH_START} → {SEARCH_END}")

## 3. Search ASF for Sentinel-1 SLC Burst Data

We query ASF's catalog for the **SLC-BURST** dataset type. This is now registered as a supported dataset (`S1_SLC_BURST`) in the `rs_tools` catalog, distinct from the OPERA RTC products.

In [ ]:
results = asf_search.search(
    dataset="SLC-BURST",
    intersectsWith=WKT,
    start=f"{SEARCH_START}T00:00:00Z",
    end=f"{SEARCH_END}T23:59:59Z",
    maxResults=250,
)

print(f"ASF returned {len(results)} burst scenes\n")

# Show a sample of the results
for r in results[:8]:
    props = r.geojson().get("properties", {})
    print(f"  {props.get('sceneName', '?'):<60}  "
          f"{props.get('startTime', '')[:19]}  "
          f"pol={props.get('polarization', '?')}")

## 4. Filter Results by Polarization for Oil Slick Detection

**VV co-polarization** is the most effective for oil slick detection:
- Oil dampens short gravity-capillary waves (Bragg scattering) → dark patches in SAR
- VV shows stronger oil-sea contrast than VH cross-polarization
- VH has higher system noise which masks the subtle backscatter reduction

We keep only **VV** bursts and group them by acquisition date to identify distinct satellite passes.

In [ ]:
# Quick summary of all VV scenes by date
vv_scenes = []
for r in results:
    props = r.geojson().get("properties", {})
    pol = props.get("polarization", "")
    start = props.get("startTime", "")
    if "VV" in pol and start:
        dt = datetime.fromisoformat(start.replace("Z", "+00:00"))
        vv_scenes.append(dt)

vv_dates = sorted(set(dt.strftime("%Y-%m-%d") for dt in vv_scenes))
print(f"VV acquisitions on {len(vv_dates)} unique dates:")
for d in vv_dates:
    flag = " ← EVENT" if d == EVENT_DATE else ""
    print(f"  {d}{flag}")

## 5. Find Same-Orbit Before/After Pair

SLC burst data is in **radar geometry** (slant-range × azimuth) and has no map CRS. Bursts from different orbits (ascending vs descending) have completely different viewing angles — they cannot be compared pixel-by-pixel.

We therefore group VV bursts by **burst ID + swath** (which uniquely identifies a radar geometry) and find the pair that brackets the event with the smallest temporal gap. This ensures valid pixel-level before/after comparison.

In [ ]:
# Group VV scenes by burst key (burst_id + swath → same radar geometry)
burst_groups = defaultdict(list)
for r in results:
    props = r.geojson().get("properties", {})
    pol = props.get("polarization", "")
    scene = props.get("sceneName", "")
    start = props.get("startTime", "")
    fdir = props.get("flightDirection", "")
    if "VV" not in pol or not start:
        continue
    dt = datetime.fromisoformat(start.replace("Z", "+00:00"))
    parts = scene.replace("-BURST", "").split("_")
    burst_key = parts[1] + "_" + parts[2]   # e.g. "343971_IW2"
    burst_groups[burst_key].append((dt, scene, fdir, r))

event_dt = datetime.fromisoformat(f"{EVENT_DATE}T00:00:00+00:00")

# Find the same-orbit pair with the smallest temporal gap bracketing the event
best_pair = None
best_gap = float("inf")

for key, entries in burst_groups.items():
    dates_sorted = sorted(entries, key=lambda x: x[0])
    before = [e for e in dates_sorted if e[0] < event_dt]
    after  = [e for e in dates_sorted if e[0] >= event_dt]
    if before and after:
        gap = (after[0][0] - before[-1][0]).days
        if gap < best_gap:
            best_gap = gap
            best_pair = (key, before[-1], after[0])

assert best_pair is not None, "No same-orbit before/after pair found!"

burst_key, before_entry, after_entry = best_pair
before_dt, before_scene, before_dir, before_obj = before_entry
after_dt, after_scene, after_dir, after_obj = after_entry
before_key = before_dt.strftime("%Y-%m-%d")
after_key = after_dt.strftime("%Y-%m-%d")

print(f"Best same-orbit pair: {burst_key} ({before_dir})")
print(f"  Before: {before_key}  {before_scene}")
print(f"  After:  {after_key}  {after_scene}")
print(f"  Gap:    {best_gap} days")

## 6. Download the Same-Orbit VV Bursts

Authenticate using the existing Earthdata `.netrc` credentials and download the two VV bursts (before and after) for the identified same-orbit pair.

In [ ]:
import netrc as _netrc

# Authenticate with Earthdata via .netrc
netrc_path = Path.home() / ".netrc"
info = _netrc.netrc(str(netrc_path))
auth = info.authenticators("urs.earthdata.nasa.gov")
assert auth, "No credentials for urs.earthdata.nasa.gov in ~/.netrc"

session = asf_search.ASFSession().auth_with_creds(auth[0], auth[2])
print("Earthdata authentication OK")


def download_burst(scene_obj, date_key, label):
    """Download a single burst TIFF; return its local path."""
    dl_dir = os.path.join(OUT_DIR, "bursts", f"{label}_{date_key}")
    os.makedirs(dl_dir, exist_ok=True)

    existing = [os.path.join(dl_dir, f)
                for f in os.listdir(dl_dir) if f.endswith((".tif", ".tiff"))]
    if existing:
        print(f"  {label}: already on disk → {existing[0]}")
        return existing[0]

    print(f"  {label}: downloading …")
    scene_obj.download(dl_dir, session=session)
    downloaded = [os.path.join(dl_dir, f)
                  for f in os.listdir(dl_dir) if f.endswith((".tif", ".tiff"))]
    print(f"  {label}: → {downloaded[0]}")
    return downloaded[0]


before_file = download_burst(before_obj, before_key, "before_same_orbit")
after_file  = download_burst(after_obj,  after_key,  "after")

## 7. Load and Preprocess Burst SLC Data

SLC burst products contain **complex-valued** radar returns ($I + jQ$).  
To get backscatter intensity we compute $|z|^2$ and convert to decibels: $\text{dB} = 10 \cdot \log_{10}(|z|^2)$.

In [ ]:
def slc_to_intensity_db(tif_path):
    """Read complex SLC → calibrated intensity in dB."""
    with rasterio.open(tif_path) as src:
        data = src.read(1)
    if np.iscomplexobj(data):
        amp = np.abs(data).astype(np.float32)
    else:
        amp = data.astype(np.float32)
    intensity = amp ** 2
    with np.errstate(divide="ignore", invalid="ignore"):
        db = 10.0 * np.log10(intensity)
    db[~np.isfinite(db)] = np.nan
    return db


before_db = slc_to_intensity_db(before_file)
after_db  = slc_to_intensity_db(after_file)

print(f"Before: {before_file}")
print(f"  shape = {before_db.shape}")
print(f"After:  {after_file}")
print(f"  shape = {after_db.shape}")

## 8. Side-by-Side Comparison

Visualise VV backscatter intensity maps for both dates using a consistent greyscale range.  
Oil slicks appear as **dark patches** (reduced backscatter) relative to surrounding water.

Both images share the same radar geometry (same orbit track and burst ID), so they can be compared pixel by pixel.

In [ ]:
# Crop to common dimensions (same-orbit bursts differ by at most 1-2 rows)
min_rows = min(before_db.shape[0], after_db.shape[0])
min_cols = min(before_db.shape[1], after_db.shape[1])
before_crop = before_db[:min_rows, :min_cols]
after_crop  = after_db[:min_rows, :min_cols]

# Common display range from percentiles
valid_before = before_crop[np.isfinite(before_crop)]
valid_after  = after_crop[np.isfinite(after_crop)]
all_valid = np.concatenate([valid_before, valid_after])
vmin = float(np.nanpercentile(all_valid, 2))
vmax = float(np.nanpercentile(all_valid, 98))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].imshow(before_crop, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
axes[0].set_title(f"Before — {before_key} (same orbit: {burst_key})", fontsize=12)
axes[0].set_axis_off()

axes[1].imshow(after_crop, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
axes[1].set_title(f"After — {after_key} (same orbit: {burst_key})", fontsize=12)
axes[1].set_axis_off()

fig.suptitle(
    f"Deurganckdock Oil Slick — Sentinel-1 VV SLC Burst (dB)\n"
    f"Same orbit track ({burst_key}, {before_dir.lower()}) "
    f"— Oil dampens surface waves → dark patch on water",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 9. Difference Map

Compute $\Delta\text{dB} = \text{after} - \text{before}$.  
Negative values (blue) indicate darkening — consistent with oil dampening wave scattering.

In [ ]:
diff = after_crop - before_crop

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(diff, cmap="RdBu_r", vmin=-5, vmax=5, origin="upper")
ax.set_title(
    f"Backscatter difference (dB): {after_key} minus {before_key}\n"
    "Blue = darker after event (potential oil slick)",
    fontsize=12,
)
ax.set_axis_off()
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="delta dB")
plt.tight_layout()
plt.show()

## 10. Interactive Before / After Slider Comparison

Using the existing `slider_comparison` module from `rs_tools.visualization` — drag the slider to compare before and after.

In [ ]:
%matplotlib widget

fig = slider_comparison(
    before_crop,
    after_crop,
    left_label=f"Before ({before_key})",
    right_label=f"After ({after_key})",
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
    title=(
        f"Deurganckdock Oil Slick — VV Backscatter (dB)\n"
        f"Before: {before_key}  |  After: {after_key}  "
        f"(same orbit: {burst_key})"
    ),
    figsize=(14, 8),
)
plt.show()

## 11. Oil Slick Analysis and Interpretation

Compute basic statistics on backscatter change and assess whether the oil slick is detectable.

**Detection conditions for SAR oil slick monitoring:**
- Wind speed 3–10 m/s is optimal (too calm → no wave contrast; too rough → oil breaks up)
- VV co-pol provides best oil-sea contrast
- Oil slicks typically reduce backscatter by 3–10 dB relative to clean water

**Note:** The burst covers a full IW sub-swath (~250 km range). The Deurganckdock is a small fraction of this area. For precise slick delineation, geocoding the SLC data would allow cropping to the dock footprint.

In [ ]:
mean_before = float(np.nanmean(valid_before))
mean_after  = float(np.nanmean(valid_after))
delta_db    = mean_after - mean_before

valid_diff = (after_crop - before_crop)
valid_diff_flat = valid_diff[np.isfinite(valid_diff)]
pct_darkened = float(np.sum(valid_diff_flat < -3.0)) / max(len(valid_diff_flat), 1) * 100

print("=" * 50)
print("  Quick Analysis — Full Burst Statistics")
print("=" * 50)
print(f"  Mean VV backscatter before:  {mean_before:.2f} dB")
print(f"  Mean VV backscatter after:   {mean_after:.2f} dB")
print(f"  Burst-wide difference:       {delta_db:+.2f} dB")
print(f"  Pixels darkened > 3 dB:      {pct_darkened:.1f}%")
print()

if delta_db < -1.0:
    print("  → Significant overall darkening detected.")
    print("    Consistent with oil dampening Bragg wave scattering,")
    print("    though part of the signal may reflect changing sea state / wind.")
elif pct_darkened > 5:
    print("  → Localised darkening detected — oil slick likely")
    print("    confined to part of the burst.")
elif delta_db < 0:
    print("  → Slight darkening — possible oil signal mixed")
    print("    with other effects (wind, tide).")
else:
    print("  → No clear darkening — oil slick may be localised")
    print("    or below detection threshold.")

print()
print("  Note: The burst covers a full IW swath (~250 km). The Deurganckdock")
print("  is a small fraction. For precise slick delineation, geocoding the")
print("  SLC data would allow cropping to the dock footprint.")